<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 4 (AI): RAG End to End — Grounding, Citations & a Real App

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Meet **LangChain** — and find the industry name for every piece you wrote by hand yesterday
2. Use **chat models and message types** (`SystemMessage`, `HumanMessage`, `AIMessage`)
3. Build reusable **prompt templates** with `ChatPromptTemplate`
4. Chain components with **LCEL**, the pipe `|` syntax
5. Turn your own `retrieve()` function into a LangChain component with **`RunnableLambda`**
6. Watch an ungrounded model **invent a fact**, confidently — then stop it
7. Make the chain answer **only** from your documents, and say **"I don't know"** when it can't
8. Return **citations** taken from chunk metadata
9. Fix retrieval when it fails: **n_results**, **distance cut-offs**, **query rewriting**, **hybrid search**, **reranking**
10. **Measure** retrieval with a golden set and hit-rate@k
11. Ship it as a **Gradio chat app** with a link you can open on your phone

> **Sections 1–7 need no API key** — the embedding model runs locally, exactly as it did yesterday.
> Only the answer-generation steps (§8 onwards) call a paid model.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai chromadb sentence-transformers langchain-text-splitters rank-bm25 gradio

In [ ]:
import os
import json
from getpass import getpass

from sentence_transformers import SentenceTransformer
import chromadb

# Configure API (only the generation steps need this)
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

# Load embedding model and initialize vector store - same two lines as yesterday
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()

print("Setup complete")

---

## 2. Yesterday by Hand, Today by Name

Yesterday you wrote every part of a retrieval pipeline yourself. Each of those parts has a standard
name and a standard interface. That is essentially what **LangChain** is:

| What you wrote yesterday | What the industry calls it | Today |
|---|---|---|
| the knowledge-base string | **Document** | text + metadata |
| `RecursiveCharacterTextSplitter` | **Text Splitter** | the same class |
| `embedder.encode(...)` | **Embeddings** | the same model |
| the Chroma collection | **Vector Store** | the same collection |
| `search(query, n_results=3)` | **Retriever** | `retrieve()`, wrapped in `RunnableLambda` |
| the system prompt + chat call | **Prompt + Chat Model** | `ChatPromptTemplate` + `ChatOpenAI` |
| gluing them together by hand | **LCEL** — the `\|` pipe | `prompt \| llm \| parser` |

Nothing new is happening. The same pieces, with names everyone recognises — and one genuinely useful
addition: because every piece speaks the *same interface*, you can swap any one of them without
touching the rest.

⚠️ **LangChain changed a lot in version 1.0.** If you find a tutorial using `RetrievalQA`,
`LLMChain` or `create_retrieval_chain`, it is written for the old version — those now live in a
separate `langchain-classic` package that is being retired. **The current way is LCEL**, which is
what we use below.

---

## 3. Chat Models

LangChain wraps LLM providers behind a unified interface. `ChatOpenAI` is the wrapper for OpenAI's
chat models.

LangChain uses three **message types** to represent a conversation:

| Message Type | Role | Purpose |
|---|---|---|
| `SystemMessage` | system | Sets the assistant's behavior and personality |
| `HumanMessage` | user | The user's input |
| `AIMessage` | assistant | The model's response |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(model=MODEL, temperature=0)

In [ ]:
# The simplest call - pass a string directly
response = llm.invoke("What is a vector database?")

print(response.content)

In [ ]:
# For more control, pass a list of message objects
messages = [
    SystemMessage(content="You are a helpful engineering tutor. Keep answers to one sentence."),
    HumanMessage(content="What is retrieval augmented generation?"),
]

response = llm.invoke(messages)

print(response.content)
print(f"\nType: {type(response)}")

In [ ]:
# The response is an AIMessage - append it to continue the conversation
messages.append(response)
messages.append(HumanMessage(content="And what problem does it solve?"))

print(llm.invoke(messages).content)

> **Key takeaway:** `ChatOpenAI` gives you one interface to the model. You talk to it with
> `SystemMessage` / `HumanMessage` / `AIMessage` objects and get an `AIMessage` back. Appending that
> `AIMessage` and a new `HumanMessage` is all "conversation memory" is — a growing list.

🧑‍🏫 Note `temperature=0`. Everything today is about the model **reading** rather than inventing.

---

## 4. Prompt Templates

Hardcoding prompts is brittle. **Prompt templates** let you define a reusable prompt with variables
that get filled in at runtime.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains {topic} concepts simply."),
    ("human", "{question}"),
])

# Inspect the template variables
print("Input variables:", prompt.input_variables)

In [ ]:
# Format the template with values - no model call yet
formatted = prompt.invoke({"topic": "search", "question": "What is an embedding?"})

print(formatted.messages)

In [ ]:
# Pass the formatted prompt to the model
print(llm.invoke(formatted).content)

> **Key takeaway:** `ChatPromptTemplate.from_messages()` separates **prompt logic** from **prompt
> data**. The same template works for any topic — which is exactly what you need when the `{context}`
> changes on every single request.

---

## 5. LCEL — chaining with the pipe

So far we passed the output of one step into the next by hand:

```python
formatted = prompt.invoke(data)
response  = llm.invoke(formatted)
```

**LCEL** lets you join them with the pipe `|` into a single chain:

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Chain: prompt -> model -> parser
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "search", "question": "What is cosine similarity?"})

print(result)
print(f"\nType: {type(result)}")

### How the pipe works

```
{"topic": "search", "question": "What is cosine similarity?"}
    |
    v
+------------------+
|  Prompt Template |  -> produces a list of messages
+------------------+
    |
    v
+------------------+
|    ChatOpenAI    |  -> produces an AIMessage
+------------------+
    |
    v
+------------------+
| StrOutputParser  |  -> produces a plain str
+------------------+
    |
    v
"Cosine similarity measures..."
```

Every component implements the **Runnable** interface with `.invoke()`. The `|` operator connects
them, so `a | b` means *"call `a.invoke()`, then pass the result to `b.invoke()`"*.

In [ ]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda wraps ANY Python function as a Runnable
def add_word_count(text):
    return f"{text}  (words: {len(text.split())})"

chain_with_count = prompt | llm | StrOutputParser() | RunnableLambda(add_word_count)

print(chain_with_count.invoke({"topic": "search", "question": "What is chunking?"}))

> **Key takeaway:** `prompt | llm | parser` is the basic chain. `RunnableLambda(fn)` turns any
> function into a link in that chain.

💡 **Hold on to `RunnableLambda`.** In section 7 it is what lets *your own* retrieval function —
the one you wrote yesterday — become a LangChain component.

---

## 6. The Knowledge Base

The same TechSolutions document you indexed yesterday — with **one addition**. Yesterday you stored
`ids`, `embeddings` and `documents`. Today each chunk also carries **metadata** saying which section
it came from, because that is what makes a citation possible later.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The same document as yesterday, split into its natural sections so every
# chunk can remember where it came from.
SECTIONS = {
    "Company Overview": """
TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma. The company is
headquartered in Bhubaneswar, Odisha, with additional offices in Bangalore and Hyderabad.
TechSolutions specializes in AI/ML solutions, cloud services, and mobile applications, and has grown
to over 250 employees. The company achieved 50 crores in annual revenue in 2024.
""",
    "Leadership": """
Priya Sharma is the CEO and co-founder. She graduated from IIT Delhi and completed her MBA from
Stanford University. She previously worked as Senior Director at Infosys and won the Women in Tech
Leader award in 2022. Rahul Verma is the CTO and co-founder. He graduated from BITS Pilani and
previously worked as Tech Lead at Google India. His expertise is in Machine Learning and Cloud
Architecture. Ananya Patel is the VP of Engineering and manages a team of 100+ engineers.
""",
    "Products": """
TechSolutions offers three main products. CloudAssist Pro is the flagship enterprise cloud
management platform, costing 50,000 per month, and includes auto-scaling, real-time monitoring,
cost optimization, and 24/7 support. SmartHR is an AI-powered HR management system at 25,000 per
month that handles recruitment automation, payroll processing, performance tracking, and employee
analytics. DataViz Analytics is a business intelligence platform at 15,000 per month with
real-time dashboards, custom reports, and predictive analytics.
""",
    "Work Policy": """
Work hours at TechSolutions are 9 AM to 6 PM, Monday to Friday, following a hybrid model with 3 days
in office and 2 days remote. Employees receive 24 paid leaves and 10 sick leaves per year. Maternity
leave is 26 weeks and paternity leave is 2 weeks. Probation period is 6 months for all new
employees, with a notice period of 2 months for permanent employees and 1 month during probation.
""",
    "Benefits and Clients": """
Employee benefits include health insurance coverage of 5 lakh for employees and their families,
covering hospitalization, OPD, dental, and vision. The learning budget is 50,000 per year per
employee for courses, certifications, and conferences. Performance bonuses can be up to 20% of
annual salary. Major clients include HDFC Bank, Tata Motors, Reliance Industries, and ICICI Bank,
with a 95% customer satisfaction rate across 200+ completed projects.
""",
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

# " ".join(t.split()) collapses the line breaks above into single spaces.
# Without it the splitter would break chunks at our line wraps instead of at
# sentence boundaries - the same "let nothing arbitrary decide your chunks"
# lesson from yesterday.
section_texts = [" ".join(t.split()) for t in SECTIONS.values()]

# create_documents attaches one metadata dict per input text - so every chunk
# inherits the section it came from.
docs = splitter.create_documents(
    section_texts,
    metadatas=[{"section": name} for name in SECTIONS],
)

print(f"{len(SECTIONS)} sections -> {len(docs)} chunks")
print()
print(docs[0].page_content)
print("metadata:", docs[0].metadata)

A LangChain **`Document`** is exactly two things: `page_content` and `metadata`. Nothing more.

💡 **Attach metadata at split time, not later.** Once chunks are embedded and stored, working out
which section a chunk came from means redoing the work.

In [ ]:
# Index it - the same three arguments as yesterday, plus metadatas
chunks = [d.page_content for d in docs]

collection = chroma.get_or_create_collection("knowledge_base_day4")
embeddings = embedder.encode(chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=[d.metadata for d in docs],
)

print(f"Indexed {collection.count()} chunks")

⚠️ **Collection names are validated** — 3–512 characters from `a-z A-Z 0-9 . _ -`, starting and
ending with a letter or number. `"kb"` or `"my docs"` will raise an error.

🧑‍🏫 We use `get_or_create_collection` rather than `create_collection` so that re-running this cell
doesn't error. In class you will re-run cells constantly.

---

## 7. Retrieval

The same function you wrote yesterday, unchanged.

In [ ]:
def retrieve(query, n_results=3):
    """Retrieve relevant documents for a query."""
    query_embedding = embedder.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    return results['documents'][0]

In [ ]:
# Change the query and re-run
query = "How much is the learning budget?"

docs_found = retrieve(query)

print(f"Query: {query}\n")
print(docs_found[0])

**Try these one at a time**, changing `query` above:

- `"Who is the CTO?"`
- `"What is the notice period?"`
- `"How many paid leaves do I get?"`
- `"Who are the major clients?"`

> **Key takeaway:** a **retriever** is any function that takes a question and returns relevant text.
> Yours is nine lines. That is all the abstraction is.

---

## 8. First, Watch It Lie

Before building the good version, see the problem clearly. Here is the model with **no context and
no rules** — just answering from what it absorbed during training.

In [ ]:
# No retrieval, no grounding - the model answering from memory
question = "What is the learning budget per employee at TechSolutions India?"

print(llm.invoke(question).content)

Read that answer carefully. TechSolutions India is a **made-up company** — the model has never seen
this document, and cannot have. Whatever number it produced, it produced from nothing.

This is the failure that makes RAG necessary, and it is worth naming precisely, because RAG systems
break in two different ways and each has a different fix:

| Failure | What happened | How you spot it | The fix |
|---|---|---|---|
| **Retrieval failure** | the right chunk was never found | print the retrieved chunks — the answer isn't in them | section 11 |
| **Generation failure** | the right chunk was found, the answer still went wrong | the answer *is* in them | section 9 |

> 🔍 **The diagnostic:** before blaming the model, *look at what you handed it.* Almost every
> "the LLM is hallucinating" bug turns out to be a retrieval bug.

---

## 9. Grounding — the chain that says "I don't know"

Now the real thing. Two rules do most of the work: **use only the context**, and **refuse when it
isn't there**.

And here is where `RunnableLambda` earns its keep — it turns your `retrieve()` function from
section 7 into a link in an LCEL chain.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for TechSolutions India.\n"
     "Answer using ONLY the context below. Do not use any other knowledge.\n"
     "If the context does not contain the answer, reply exactly: "
     "I don't know based on the company documents.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    """Turn a list of retrieved chunks into one plain string for the prompt."""
    return "\n\n".join(docs)

In [ ]:
# YOUR retrieve() function, now a LangChain component
retriever = RunnableLambda(retrieve)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the learning budget per employee?"))

Compare that with section 8. Same question, same model — the only thing that changed is that it was
handed the right paragraph and told to stay on it.

Read the chain top to bottom:

```
              "What is the learning budget per employee?"
                             |
      +----------------------+----------------------+
      |                                             |
  retriever                                RunnablePassthrough
  (your retrieve fn)                       (keep the question as-is)
      |                                             |
  format_docs                                       |
      |                                             |
      +-----------------> RAG_PROMPT <--------------+
                              |
                             llm          temperature=0
                              |
                       StrOutputParser
                              |
                           answer
```

**The dict at the front is the trick worth remembering:** each key is computed **in parallel**, and
the result is exactly the `{context}` and `{question}` the prompt is asking for.

💡 Notice `format_docs` is a plain function, not wrapped in `RunnableLambda`. Inside a pipe LangChain
coerces functions automatically — `RunnableLambda` is only needed when you want the Runnable itself.

In [ ]:
# Something the document simply does not contain
print(rag_chain.invoke("Who is the CFO of TechSolutions?"))

The document names a CEO, a CTO and a VP of Engineering — **there is no CFO in it.**

💡 **That refusal is the feature.** Retrieval on its own has no notion of "nothing here is relevant" —
it always returns its top-k. The *prompt* is what turns "here are the 3 closest chunks" into "this
document doesn't answer your question."

A system that says "I don't know" 5% of the time is worth far more than one that is confidently wrong
5% of the time — because nobody can tell which 5% they're reading.

> **Key takeaway:** grounding = *use only the context* + *refuse in fixed wording*. The fixed wording
> matters: your code can detect it, log it, and route the user somewhere useful.

---

## 10. Citations

An answer nobody can verify is an answer they have to take on faith. Citations fix that — and they
come from the **metadata**, never from the model.

In [ ]:
def answer_with_sources(question, n_results=3):
    """Retrieve, answer from context, and report where the text came from."""
    query_embedding = embedder.encode([question])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    context = "\n\n".join(results['documents'][0])
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return answer, results['metadatas'][0], results['distances'][0]

In [ ]:
answer, sources, distances = answer_with_sources("What is the notice period for permanent employees?")

print(answer)
print("\nSources:")
for meta, dist in zip(sources, distances):
    print(f"  - {meta['section']}  (distance {dist:.3f})")

⚠️ **Never ask the model to produce the citation.** A model asked to "cite your source" will happily
invent a plausible-looking reference — exactly the behaviour you're trying to eliminate. The section
name above was attached at split time, stored with the chunk, and printed by **your** code. The model
never touched it.

💡 The `distances` come free from Chroma, and **lower is better** (yesterday's point). They are your
confidence signal — we use them in section 11.

> **Key takeaway:** the answer is **generated**; the citation is **retrieved**. Two completely
> different trust levels.

---

## 11. When Retrieval Fails — the fix kit

Grounding stops the model inventing. It does **not** help when the right chunk was never retrieved —
then a well-behaved system politely says "I don't know" about something that is in the document.

Start every investigation the same way: **look at what was retrieved.**

In [ ]:
# The habit that saves hours: inspect the chunks before blaming the model
question = "What is the learning budget?"

for i, doc in enumerate(retrieve(question), 1):
    print(f"[{i}] {doc[:110]}...")
    print()

### 11.1 Turn the knob — `n_results`

The cheapest fix first: retrieve **more**.

In [ ]:
# Change n_results and re-run
docs_found = retrieve("What is the learning budget?", n_results=6)

print(f"{len(docs_found)} chunks retrieved")
print(docs_found[-1][:150])

More chunks means better odds of catching the right one — and a longer, costlier, noisier prompt.
Research on ["Lost in the Middle"](https://arxiv.org/abs/2307.03172) shows models attend best to the
**start and end** of their context, so burying the good chunk among ten others can make things
*worse*. Retrieve broad, then narrow — that's reranking, in 11.4.

### 11.2 A distance cut-off — refusing before you pay

You already have `distances` from Chroma, and yesterday's rule holds: **lower is closer**. If even the
best chunk is far away, there is no point calling the model at all.

**But don't guess the cut-off — look at your data first.**

In [ ]:
# The closest distance for a question the document answers, and one it doesn't
def best_distance(question):
    results = collection.query(
        query_embeddings=embedder.encode([question]).tolist(),
        n_results=1
    )
    return results['distances'][0][0]


print(f"in  the document : {best_distance('How much is the learning budget?'):.3f}")
print(f"not in the doc   : {best_distance('What is the capital of Brazil?'):.3f}")

Two numbers, and your cut-off goes in the gap between them.

🧑‍🏫 **Why the numbers look the way they do.** `all-MiniLM-L6-v2` returns **unit-length** vectors, and
Chroma's default space is **squared L2**. That makes the two measures exactly related:

```
distance = 2 - 2 x cosine_similarity        ->        cosine = 1 - distance / 2
```

So a distance of `1.2` is a cosine of `0.4`. It is the same geometry as yesterday, reported on a
different scale — not a new concept to memorise.

In [ ]:
def retrieve_or_none(question, max_distance=1.2, n_results=3):
    """Return chunks only if the best one is close enough. Lower distance = closer."""
    results = collection.query(
        query_embeddings=embedder.encode([question]).tolist(),
        n_results=n_results
    )

    if results['distances'][0][0] > max_distance:
        return None

    return results['documents'][0]

In [ ]:
# Change the question and re-run - try an in-document one, then something unrelated
question = "What is the capital of Brazil?"

found = retrieve_or_none(question)

print(f"Query: {question}")
print("Retrieved:" if found else "Nothing relevant - skipping the model call entirely")

⚠️ **`max_distance=1.2` is a starting point, not a law.** Distance bands are model- and data-specific —
the same warning as yesterday's similarity thresholds. Set it from the two numbers you just printed,
then tune it against the golden set in section 12. Too tight and you refuse answerable questions; too
loose and it does nothing at all.

🧑‍🏫 The grounding prompt already handles irrelevant context gracefully. The cut-off's real value is
**cost and latency** — it lets you skip the API call altogether.

### 11.3 Query rewriting

Users don't type search queries. They type *questions* — vague, conversational, full of pronouns.

In [ ]:
REWRITE_PROMPT = ChatPromptTemplate.from_template(
    "Rewrite the question as a short, keyword-rich search query for a company knowledge base.\n"
    "Return ONLY the query.\n\n"
    "Question: {question}"
)
rewriter = REWRITE_PROMPT | llm | StrOutputParser()

vague = "what do I get if I join?"

print("Original: ", vague)
print("Rewritten:", rewriter.invoke({"question": vague}))

In [ ]:
# Does the rewrite retrieve better? Compare the first chunk each one finds.
print("raw       ->", retrieve(vague)[0][:90])
print()
print("rewritten ->", retrieve(rewriter.invoke({"question": vague}))[0][:90])

### 11.4 Hybrid search  `[Extended]`

Vector search is strong on meaning and weak on **exact strings**. Ask it for "HDFC" and it may hand
you paragraphs *about* clients that never contain the name. Keyword search (BM25) has the opposite
strengths — so production systems run both and merge the rankings.

In [ ]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi([c.lower().split() for c in chunks])


def hybrid_search(query, n_results=3):
    """Merge two rankings with Reciprocal Rank Fusion: score = sum of 1 / (60 + rank)."""
    scores = {}

    # ranking 1 - meaning
    vector_ids = collection.query(
        query_embeddings=embedder.encode([query]).tolist(), n_results=10
    )['ids'][0]
    for rank, cid in enumerate(vector_ids):
        idx = int(cid.split("_")[1])
        scores[idx] = scores.get(idx, 0) + 1 / (60 + rank)

    # ranking 2 - exact words
    for rank, idx in enumerate(bm25.get_scores(query.lower().split()).argsort()[::-1][:10]):
        scores[idx] = scores.get(idx, 0) + 1 / (60 + rank)

    best = sorted(scores, key=scores.get, reverse=True)[:n_results]
    return [chunks[i] for i in best]

In [ ]:
# A query with an exact term in it - change it and re-run
query = "HDFC"

print("vector only ->", retrieve(query, n_results=1)[0][:100])
print()
print("hybrid      ->", hybrid_search(query, n_results=1)[0][:100])

💡 **Reciprocal Rank Fusion** needs no tuning and no score normalisation — it only cares *what
position* each result reached in each list. A chunk both methods rank highly wins; a chunk only one
method loves still gets a chance. The `60` is a standard constant that stops rank 1 dominating.

### 11.5 Reranking  `[Extended]`

The embedding model encoded every chunk **before it ever saw your question** — that is what makes
search fast, and also what makes it approximate. A **reranker** looks at the question and a chunk
*together* and scores that pair properly.

```
retrieve 6 (fast, approximate)  ->  rerank (slow, accurate)  ->  keep top 3  ->  LLM
```

In [ ]:
RERANK_PROMPT = ChatPromptTemplate.from_template(
    "Question: {question}\n\n"
    "Documents:\n{docs}\n\n"
    "Reorder the documents from most to least relevant to the question.\n"
    "Return ONLY a JSON array of document numbers, for example [3, 1, 2]."
)
reranker = RERANK_PROMPT | llm | StrOutputParser()

question = "Who has an advanced degree?"

# Retrieve broad...
candidates = retrieve(question, n_results=6)
listing = "\n".join(f"[{i + 1}] {d}" for i, d in enumerate(candidates))

# ...then let the model put them in order
order = json.loads(reranker.invoke({"question": question, "docs": listing}))

print("reranked order:", order)
print()
print("new top chunk:", candidates[order[0] - 1][:150])

🧑‍🏫 **In production you would not use an LLM for this.** A dedicated **cross-encoder** is 10–100×
faster and cheaper: [`cross-encoder/ms-marco-MiniLM-L-6-v2`](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)
is free and runs locally (same library as our `embedder`), and
[Cohere Rerank](https://docs.cohere.com/docs/rerank) is the common hosted option. We use the LLM here
because you already have the key — the *idea* is identical.

### The fix kit, summarised

| Symptom | Reach for | Cost |
|---|---|---|
| right chunk just missed the cut | bigger `n_results` | longer prompt |
| question is vague or conversational | **query rewriting** | +1 LLM call |
| exact terms, codes, names missed | **hybrid search** | none |
| right chunk retrieved but ranked 5th | **reranking** | +1 call / +latency |
| nothing relevant exists | **distance cut-off** + the grounding prompt | none |
| three chunks all say the same thing | **MMR** (a diversity-aware ranking) | none |

---

## 12. Measure It

Every knob above is a guess until you measure it. The cheapest useful measurement in RAG: take a
handful of real questions, write down a fact that **must** appear in the retrieved chunks, and count
how often it does.

That's a **golden set**, and six questions written in ten minutes will teach you more than any amount
of staring at the code.

In [ ]:
# A golden set: a real question + a fact the retrieved chunks MUST contain
GOLDEN = [
    ("How much is the learning budget?",              "50,000"),
    ("Who is the CTO?",                               "Rahul Verma"),
    ("What is the notice period for permanent staff?", "2 months"),
    ("How many paid leaves per year?",                "24 paid leaves"),
    ("What is the health insurance coverage?",        "5 lakh"),
    ("Who are the major clients?",                    "HDFC"),
]


def hit_rate(search_fn, n_results=3):
    """Fraction of questions whose expected fact appears somewhere in the retrieved chunks."""
    hits = 0
    for question, expected in GOLDEN:
        retrieved = " ".join(search_fn(question, n_results)).lower()
        hits += expected.lower() in retrieved
    return hits / len(GOLDEN)

In [ ]:
# One number for the whole retrieval step. Change n_results and re-run.
n = 3

print(f"hit rate @{n}: {hit_rate(retrieve, n_results=n):.0%}")

Now you can answer questions that were previously matters of opinion:

- Does `n_results=1` still work? (drop `n` and re-run)
- Is hybrid search better *here*? (pass `hybrid_search` instead of `retrieve`)
- Was `chunk_size=200` a good choice? (change it in section 6, re-run everything below it)

🧑‍🏫 **This is the honest answer to "what chunk size should I use?"** — you measure it. Not vibes,
not a blog post. And notice this measures **retrieval only**: it says nothing about whether the final
*answer* was good. Judging answer quality (LLM-as-judge, faithfulness, RAGAS) is Day 5.

⚠️ A golden set of 6 is a teaching size. Aim for 30–100 real user questions before you trust the number.

---

## 13. Ship It — a chat app

A notebook cell is not a product. The last step is a chat interface — and chat introduces one
genuinely new problem: **follow-up questions don't retrieve well on their own.**

> "Who is the CTO?" → *Rahul Verma*
> "**Where did he study?**" ← embed *that* and you will retrieve nothing useful.

The fix is the rewriter from 11.3, now given the conversation so far.

In [ ]:
def rag_answer(message, history):
    """One chat turn: resolve the follow-up, retrieve, ground, then cite."""

    # 1) Turn a follow-up into a standalone search query using what was said before
    if history:
        recent = "\n".join(f"{m['role']}: {m['content']}" for m in history[-4:])
        search_query = rewriter.invoke({"question": f"{recent}\nFollow-up: {message}"})
    else:
        search_query = message

    # 2) Retrieve on the rewritten query...
    results = collection.query(
        query_embeddings=embedder.encode([search_query]).tolist(),
        n_results=3
    )

    # 3) ...but answer the question the user actually asked
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(results['documents'][0]), "question": message}
    )

    # 4) Cite from metadata
    sections = sorted({m['section'] for m in results['metadatas'][0]})
    return f"{answer}\n\n*Source: {', '.join(sections)}*"

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    rag_answer,
    type="messages",
    title="TechSolutions AI Assistant",
    description="Ask about the company, products, policies and benefits. Answers come only from the company documents.",
    examples=[
        "How much is the learning budget?",
        "What is the work from home policy?",
        "What products does the company offer?",
        "Who is the CFO?",
    ],
)

demo.launch(share=True)

Open the `gradio.live` link on your phone. That is a working document assistant, and every part of it
is something you built.

**Try the last example — "Who is the CFO?"** It is in the list deliberately. The documents name a
CEO, a CTO and a VP of Engineering, and a good assistant should say so rather than guess.

Then try a follow-up: ask *"Who is the CTO?"*, and then just *"Where did he study?"* Watch the
rewriter turn that into a real search query.

💡 **The subtle bit:** retrieve on the **rewritten** query, answer the question the user **actually
asked**. Mix those two up and your bot starts answering questions nobody asked.

---

## 14. Exercises

Fill in the blanks (`___`) and run each cell.

### Q1: Your first LCEL chain

Build a three-step chain that turns a topic into a one-line summary.

In [ ]:
# Hint: the pipe order is always prompt | llm | parser.
#       ChatPromptTemplate.from_template uses {braces} for variables.

tip_prompt = ChatPromptTemplate.from_template("Summarise {topic} in one short sentence.")

tip_chain = tip_prompt | ___ | StrOutputParser()

print(tip_chain.invoke({"___": "vector databases"}))

### Q2: Retrieve and read the metadata

Retrieve for a question of your own and print which section each chunk came from.

In [ ]:
# Hint: collection.query(query_embeddings=, n_results=) returns lists of lists - index [0].
#       Each metadata dict looks like {"section": "Products"}

my_question = "How much does SmartHR cost?"

results = collection.query(
    query_embeddings=embedder.encode([my_question]).___(),
    n_results=3,
)

print("sections:", [m["___"] for m in results['metadatas'][0]])
print(results['documents'][0][0])

### Q3: Make it refuse

Wrap your own `retrieve` in a chain, then ask it something the documents do not cover.

In [ ]:
# Hint: RunnableLambda(fn) turns a function into a Runnable.
#       The two rules are "use ONLY the context" and "say you don't know otherwise".

MY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. If it is not there, say you don't know.\n\n{context}"),
    ("human", "{question}"),
])

my_chain = (
    {"context": ___(retrieve) | format_docs, "question": RunnablePassthrough()}
    | MY_PROMPT
    | llm
    | ___
)

print(my_chain.invoke("How many sick leaves per year?"))   # answerable
print(my_chain.invoke("___"))                              # ask something NOT in the documents

### Q4: Measure a change

Add a question of your own to the golden set, then compare two values of `n_results`.

In [ ]:
# Hint: hit_rate(search_fn, n_results) - the search_fn takes (query, n_results).

GOLDEN.append(("What is the maternity leave?", "___"))   # a fact that must be retrieved

low  = hit_rate(retrieve, n_results=___)
high = hit_rate(retrieve, n_results=___)

print(f"low:  {low:.0%}")
print(f"high: {high:.0%}")

### Q5: Cite your sources

Return an answer together with the sections it came from.

In [ ]:
# Hint: answer_with_sources(question) returns (answer, sources, distances).

answer, sources, distances = ___("What is the probation period?")

print(answer)
print("Cited sections:", sorted({m["___"] for m in sources}))

---

## Key Takeaways

### Concept Map

```
ChatOpenAI(model="gpt-4o-mini", temperature=0)   --- the LLM wrapper
  |
ChatPromptTemplate.from_messages([...])          --- reusable prompt with {variables}
  |
LCEL (the pipe)                                  --- chain components together
  |
  |-- prompt | llm | StrOutputParser()           --- the basic chain
  |-- RunnableLambda(retrieve)                   --- YOUR function as a component
  |-- RunnablePassthrough()                      --- pass the input through unchanged
  |
retrieve(query, n_results)                       --- embed -> collection.query -> top-k
  |
  |-- results['documents'][0]                    --- the chunk text
  |-- results['metadatas'][0]                    --- where it came from  -> citations
  |-- results['distances'][0]                    --- how close  (lower is better)
```

### Quick Reference

| Idea | The one-liner |
|---|---|
| **LCEL** | `prompt \| llm \| parser` — every component speaks `.invoke()`, so they pipe together |
| **`RunnableLambda`** | wraps any function as a chain link — including your own `retrieve()` |
| **`RunnablePassthrough`** | "this key is just the input, unchanged" |
| **Document** | `page_content` + `metadata`; the metadata is what makes citation possible |
| **Grounding** | "use ONLY the context" + "say you don't know" — two rules, most of the value |
| **`temperature=0`** | for RAG you want the model reading, not inventing |
| **Citations** | come from *your* metadata; a model asked to cite will invent a reference |
| **The diagnostic** | print the retrieved chunks before blaming the model |
| **Retrieval failure** | right chunk never found → `n_results`, rewriting, hybrid, reranking |
| **Generation failure** | right chunk found, answer still wrong → fix the prompt |
| **`distances`** | lower is better; a cut-off lets you refuse before paying for a call |
| **Golden set** | a few real questions + expected facts = the only honest way to tune |

### 🏠 Homework

1. **Swap the document.** Replace `SECTIONS` with something you actually care about — your notes, a
   syllabus, a product manual. Everything below section 6 should work unchanged.
2. **Write a golden set of 10** questions for your document, and record the hit rate at
   `n_results=1`, `3` and `5`.
3. **Break it on purpose.** Find one question where retrieval fails, then fix it with exactly one
   technique from section 11 — and write down which one and why.

### 📚 Resources

- [LangChain — LCEL and Runnables](https://python.langchain.com/docs/concepts/lcel/)
- [LangChain — retrievers](https://python.langchain.com/docs/concepts/retrievers/)
- [Lost in the Middle (Liu et al., 2023)](https://arxiv.org/abs/2307.03172) — why position in the context matters
- [Query Rewriting for Retrieval-Augmented LLMs (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)
- [Cohere Rerank](https://docs.cohere.com/docs/rerank) · [free local cross-encoder](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)
- [RAGAS](https://docs.ragas.io/) — evaluation at production scale *(Day 5)*

---

**Next — Day 5:** agents that decide *which* tool to use and when, plus production GenAI: cost,
evaluation and responsible AI.